---

# Submission Kaggle: SVM Kernel Laplacian dan DTL

*Prediksi pembatalan pesanan hotel dengan model from scratch, tanpa ensemble.*

---

## Kelompok G04: TreeIsAllYouNeed

---

## Daftar Isi

1. [**Pendahuluan**](#1)
2. [**Inisialisasi**](#2)
3. [**Data Cleaning dan Transformation**](#3)
4. [**Model 1: SVM Kernel Laplacian**](#4)
5. [**Model 2: DTL C4.5**](#5)
6. [**Referensi**](#6)

---

# Pendahuluan <a name="1"></a>

---

Notebook ini melatih dua model dari data mentah lalu menulis dua berkas submission. Tiap model berdiri
sendiri. Prediksinya tidak digabung atau dirata-rata.

| Model | Berkas | Skor publik Kaggle | Macro F1, CV 5-fold |
|---|---|---|---|
| SVM kernel Laplacian | `output/kaggle_final/svm/submission.csv` | 0,87464 | 0,8703 (7 seed), 0,8699 (4 seed konfirmasi) |
| DTL C4.5 | `output/kaggle_final/dtl/submission.csv` | 0,87393 | 0,8672 (10 seed) |

Pada CV, seluruh fitur dan bobot fitur dihitung ulang di dalam setiap fold. CV untuk SVM memakai
implementasi SMO yang setara dengan toleransi 0,01 agar cepat, sedangkan notebook ini memakai class `SVM`
dengan toleransi 0,001. Kedua toleransi itu menghasilkan label test yang sama.

Seluruh notebook selesai dalam sekitar 1 sampai 10 menit, tergantung memori yang tersedia.

---

# Inisialisasi <a name="2"></a>

---

Pustaka yang dipakai hanya NumPy dan pandas. Algoritma SVM dan DTL diimpor dari `src/algoritma`.

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "kaggle" / "dataset" / "train.csv").exists())
for folder in ("algoritma", "algoritma/svm", "kaggle"):
    sys.path.insert(0, str(REPO / "src" / folder))

from dtl import DTL
from kaggle_final_features import build_features, clean, feature_weights, load_data, standardize
from svm import SVM

DATA = REPO / "src" / "kaggle" / "dataset"
OUTPUT = REPO / "output" / "kaggle_final"
print("numpy", np.__version__, " pandas", pd.__version__)

numpy 2.5.4  pandas 3.1.1


In [2]:
train_raw, test = load_data(DATA)
sample = pd.read_csv(DATA / "sample_submission.csv", dtype={"id": str})
print("train:", train_raw.shape, " test:", test.shape)


def write_submission(labels, folder):
    submission = pd.DataFrame({"id": test["id"], "is_canceled": labels})
    assert list(submission.columns) == list(sample.columns)
    assert submission["id"].tolist() == sample["id"].tolist()
    folder.mkdir(parents=True, exist_ok=True)
    submission.to_csv(folder / "submission.csv", index=False)
    print("baris:", len(submission), " rasio batal:", round(labels.mean(), 4))

train: (27373, 19)  test: (6844, 18)


---

# Data Cleaning dan Transformation <a name="3"></a>

---

Duplikat persis pada train dibuang. Nilai kosong diisi median atau modus train, disertai penanda kosong.
Fitur tambahan terdiri atas tanggal booking, lama menginap, dan fitur saudara, yaitu rata-rata label booking
lain yang kemungkinan satu rombongan. Saudara harus memiliki tanggal datang, jumlah malam, dan kanal yang
sama persis, selisih lead time paling banyak 10 hari, selisih harga paling banyak 5%, serta kolom lunak yang
sama atau kosong.

Untuk baris train, fitur saudara dihitung out-of-fold sehingga label baris itu sendiri tidak terlihat.
Semua statistik berasal dari train. Train dan test tidak digabung, dan label test tidak dipakai.

In [3]:
train = clean(train_raw)
y = train["is_canceled"].to_numpy()
X_snn, X_snn_test = build_features(train, y, test, shared_neighbours=True)
X_dtl, X_dtl_test = build_features(train, y, test, shared_neighbours=False)
print("train:", len(train_raw), "->", len(train), "baris, fitur:", X_snn.shape[1])
print("sisa nilai kosong di test:", int(X_snn_test.isna().sum().sum()))

train: 27373 -> 27309 baris, fitur: 46
sisa nilai kosong di test: 0


---

# Model 1: SVM Kernel Laplacian <a name="4"></a>

---

Kernel Laplacian memakai jarak L1, yaitu K(a, b) = exp(-jumlah(|a - b|) / 20). Jarak L1 menjumlahkan
selisih per fitur, sehingga lebih cocok untuk data tabular daripada jarak L2 pada kernel Gaussian.

## Bobot Fitur

Setiap fitur diberi bobot sebelum masuk ke kernel. Bobot dihitung dari train dengan dua putaran: regresi
kernel ridge dilatih, lalu bobot fitur diambil dari besar gradien fungsinya terhadap fitur tersebut. Fitur
yang tidak memengaruhi prediksi mendapat bobot kecil.

In [4]:
SCALE = 20.0
A, B, names = standardize(X_snn, X_snn_test)
started = time.time()
weights = feature_weights(A, y, SCALE)
print(f"selesai dalam {time.time() - started:.0f} detik")
pd.Series(weights, index=names).sort_values(ascending=False).head(12)

selesai dalam 11 detik


sib_te                                 3.270
days_before_arrival                    2.790
book_ord                               2.303
nightly_rate_idr                       2.190
arr_ord                                1.919
special_requests                       1.556
arrival_day                            1.293
arrival_month                          1.180
sib_n                                  1.131
weekend_share                          0.813
booking_channel_Online Travel Agent    0.803
arr_dow                                0.772
dtype: float64

## Pelatihan, Save, dan Load

SVM dilatih dengan SMO dan C = 2. Model disimpan, dimuat kembali, lalu model hasil muat itulah yang dipakai
untuk prediksi. Booking diprediksi batal jika nilai keputusannya positif.

In [5]:
started = time.time()
svm = SVM(kernel="laplacian", gamma=1 / SCALE, C=2.0, tol=1e-3, cache_mb=400).fit(A * weights, y)
print(f"iterasi SMO: {svm.n_iter_}, support vector: {len(svm.support_)}, {time.time() - started:.0f} detik")

(OUTPUT / "svm").mkdir(parents=True, exist_ok=True)
svm.save(OUTPUT / "svm" / "svm_model.pkl")
svm = SVM.load(OUTPUT / "svm" / "svm_model.pkl")

iterasi SMO: 22302, support vector: 8662, 29 detik


In [6]:
labels = svm.predict(B * weights)
write_submission(labels, OUTPUT / "svm")

baris: 6844  rasio batal: 0.2984


---

# Model 2: DTL C4.5 <a name="5"></a>

---

Pohon memakai gain ratio dan cost-complexity pruning. Fitur saudaranya memakai bobot kedekatan lead time
saja. Booking diprediksi batal jika P(batal) minimal 0,4.

In [7]:
tree = DTL(max_depth=20, min_samples_leaf=2, pruning="cost_complexity", ccp_alpha=2.5).fit(X_dtl, y)
print("daun:", tree.n_leaves_, " kedalaman:", tree.depth_)

(OUTPUT / "dtl").mkdir(parents=True, exist_ok=True)
tree.save(OUTPUT / "dtl" / "dtl_model.json")
tree = DTL.load(OUTPUT / "dtl" / "dtl_model.json")

labels = (tree.predict_proba(X_dtl_test)[:, 1] >= 0.4).astype(int)
write_submission(labels, OUTPUT / "dtl")

daun: 82  kedalaman: 20
baris: 6844  rasio batal: 0.3025


---

# Referensi <a name="6"></a>

---

1. B. Scholkopf dan A. Smola, *Learning with Kernels*, MIT Press, 2002.
2. J. Platt, "Sequential Minimal Optimization", MSR-TR-98-14, 1998.
3. R.-E. Fan, P.-H. Chen, dan C.-J. Lin, "Working Set Selection Using Second Order Information", JMLR 6, 2005.
4. A. Radhakrishnan dkk., "Mechanism for feature learning in neural networks and backpropagation-free machine
   learning models", Science 383, 2024.
5. C. Williams dan M. Seeger, "Using the Nystrom Method to Speed Up Kernel Machines", NIPS 13, 2001.
6. D. Micci-Barreca, SIGKDD Explorations 3(1), 2001.
7. R. Jarvis dan E. Patrick, IEEE Transactions on Computers C-22, 1973.
8. J. R. Quinlan, *C4.5: Programs for Machine Learning*, 1993, dan L. Breiman dkk., *Classification and
   Regression Trees*, 1984.